# Run the pipeline on Colab, Kaggle or locally

Runs the project's real CLI (`python -m src.cli ...`); nothing here re-implements the pipeline.

* **SMOKE = True** (default): 3-channel Telemanom-aligned run, a few minutes. Checks the setup only, not a result.
* **SMOKE = False**: full 81-channel Telemanom-aligned run under a new run name. It is resumable: rerun the
  notebook after a disconnect and finished channels are skipped.
* Data: an attached Kaggle copy is used if found (every file is checked against the committed checksums);
  otherwise the verified downloader fetches it (~80 MB).
* Outputs are zipped into `OUTPUT_DIR` (`/kaggle/working`, Google Drive if mounted, or `outputs/` locally).

Verification status: executed locally (CPU, smoke mode) by `tests/test_notebook.py`. **It has not been run on
Colab or Kaggle, and GPU speed-ups are not measured.** Training uses CUDA when `DEVICE = "auto"` finds a GPU;
scoring always runs on CPU.

In [ ]:
SMOKE = True          # False = full 81-channel run (hours on CPU)
DEVICE = "auto"       # "auto" -> CUDA if available, else CPU
SEED = 42
REPO_URL = "https://github.com/SJ-Hariharan/nasa-telemetry-anomaly-detection"


In [ ]:
import os, subprocess, sys
from pathlib import Path

# Colab images also contain /kaggle, so Kaggle is detected from its runtime variable, never from paths.
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = not IN_COLAB and bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if IN_COLAB or IN_KAGGLE:
    root = Path("/content" if IN_COLAB else "/kaggle/working") / "nasa-telemetry-anomaly-detection"
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "requirements.txt")], check=True)
else:   # local: this notebook lives in <repo>/notebooks
    root = Path.cwd().resolve()
    root = root if (root / "src" / "cli.py").exists() else root.parent
os.chdir(root)
print("repository:", root)

def cli(*args):
    print("$ python -m src.cli", " ".join(args), flush=True)
    r = subprocess.run([sys.executable, "-m", "src.cli", *args], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-2000:] if r.returncode else "", sep="\n")
    r.check_returncode()


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


In [ ]:
raw = root / "data" / "raw" / "telemanom"
if (raw / "labeled_anomalies.csv").exists():
    print("data already present")
elif IN_KAGGLE and Path("/kaggle/input").exists():
    cli("download", "--from-dir", "/kaggle/input")   # fails loudly if the attached copy differs from the benchmark
else:
    cli("download")
cli("validate")


In [ ]:
run = "telemanom_nb_smoke" if SMOKE else f"telemanom_nb_seed{SEED}"
base = "telemanom_smoke.yaml" if SMOKE else "telemanom.yaml"
cfg = root / "configs" / f"{run}.yaml"
cfg.write_text(f"inherit: {base}\nrun_name: {run}\nseed: {SEED}\nmethods: [telemanom]\n"
               f"telemanom:\n  device: {DEVICE}\n")
cli("train", "--config", str(cfg.relative_to(root)), "--resume")
cli("evaluate", "--config", str(cfg.relative_to(root)))


In [ ]:
import json, shutil, time
s = json.loads((root / "reports/runs" / run / "training_summary.json").read_text())
print(f"{s['n_channels']} channels, {s['total_seconds']:.0f} s, failures: {s['failures'] or 'none'}")
print("device requested:", {c['methods']['telemanom'].get('device') for c in s['channels']},
      "| device used for training:", {c['methods']['telemanom'].get('train_device') for c in s['channels']})
print((root / "reports/runs" / run / "comparison.md").read_text())

drive = Path("/content/drive/MyDrive")
OUTPUT_DIR = Path("/kaggle/working") if IN_KAGGLE else (drive if drive.exists() else root / "outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
archive = shutil.make_archive(str(OUTPUT_DIR / f"{run}_{time.strftime('%Y%m%d-%H%M')}"), "zip",
                              root / "reports" / "runs", run)
print("saved", archive)
